# Dynamic Workers — Demo Notebook

This notebook demonstrates how to interact with the **Dynamic Worker Loaders** API.

The host Worker exposes a `POST /api/run` endpoint. You send it a snippet of Cloudflare Worker JavaScript code, it spins up a sandboxed, disposable isolate, runs it, and returns the response.

**Start the dev server first:**
```sh
npm start   # from the project root
```
The server listens on `http://localhost:5173` by default.

In [ ]:
import json
import requests

BASE_URL = "http://localhost:5173"

def run_worker(code: str) -> dict:
    """Send code to /api/run and return the parsed JSON response."""
    resp = requests.post(f"{BASE_URL}/api/run", json={"code": code})
    return resp.json()

## Example 1 — Hello World

The simplest possible dynamic Worker: return a plain-text greeting.

In [ ]:
code = """
export default {
  fetch() {
    return new Response("Hello from a dynamic Worker!");
  },
};
"""

result = run_worker(code)
print(json.dumps(result, indent=2))

## Example 2 — JSON Response

Return structured JSON data from the isolate.

In [ ]:
code = """
export default {
  fetch(request) {
    return Response.json({
      message: "Dynamic isolate reporting in!",
      timestamp: Date.now(),
      url: request.url,
    });
  },
};
"""

result = run_worker(code)
print(json.dumps(result, indent=2))

# Parse the nested JSON output
if result.get("ok"):
    payload = json.loads(result["output"])
    print("\nParsed payload:", payload)

## Example 3 — Computation in the Isolate

Run arbitrary logic — here, generating the first N Fibonacci numbers.

In [ ]:
code = """
export default {
  fetch() {
    const n = 10;
    const fibs = [0, 1];
    for (let i = 2; i < n; i++) {
      fibs.push(fibs[i - 1] + fibs[i - 2]);
    }
    return Response.json({ fibonacci: fibs });
  },
};
"""

result = run_worker(code)
if result.get("ok"):
    payload = json.loads(result["output"])
    print("Fibonacci sequence:", payload["fibonacci"])

## Example 4 — Custom HTTP Status

The isolate can return any HTTP status code; the host forwards it.

In [ ]:
code = """
export default {
  fetch() {
    return new Response("Created!", { status: 201 });
  },
};
"""

result = run_worker(code)
print(f"ok={result['ok']}  status={result.get('status')}  output={result.get('output')!r}")

## Example 5 — Error Handling

When the submitted code is invalid or throws, the API returns `ok: false` with an error message.

In [ ]:
bad_code = """
this is not valid JavaScript !!!!
"""

result = run_worker(bad_code)
print(json.dumps(result, indent=2))

## Example 6 — Network Isolation

Dynamic isolates are created with `globalOutbound: null`, which blocks all outbound network access. Any `fetch()` call inside the isolate will fail.

In [ ]:
code = """
export default {
  async fetch() {
    try {
      await fetch("https://example.com");
      return new Response("Fetch succeeded (unexpected)");
    } catch (err) {
      return new Response(`Blocked as expected: ${err.message}`, { status: 403 });
    }
  },
};
"""

result = run_worker(code)
print(f"status={result.get('status')}")
print(result.get("output"))

## Batch Runner

Send multiple snippets in a loop and collect results — useful for automated testing.

In [ ]:
snippets = {
    "plain text": 'export default { fetch() { return new Response("plain"); } };',
    "json body":  'export default { fetch() { return Response.json({ x: 42 }); } };',
    "404 status": 'export default { fetch() { return new Response("nope", { status: 404 }); } };',
}

for name, code in snippets.items():
    r = run_worker(code)
    status_str = f"HTTP {r.get('status')}" if r.get("ok") else f"ERROR: {r.get('error')}"
    print(f"{name:<14}  {status_str:<12}  output={r.get('output')!r}")